# Import packages and  Load  Env Variables

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from dotenv import load_dotenv
import os

# Load environment variables from the .env file
load_dotenv()

# Access the environment variables
api_key = os.getenv('api_key')
access_token = os.getenv('access_token')


# Define Parameters

In [147]:
# Define the list of movie IDs
movie_ids = [
    0, 299534, 19995, 140607, 299536, 597, 135397, 420818, 24428, 
    168259, 99861, 284054, 12445, 181808, 330457, 351286, 109445, 
    321612, 260513
]

# TMDb API URL and your access token (replace with your actual token)
api_url = "https://api.themoviedb.org/3/movie/{}"


# Fetch Data

In [ ]:
# Function to fetch movie data from TMDb API
def fetch_movie_data(movie_id):
    url = api_url.format(movie_id)
    headers = {
        'Authorization': f'Bearer {access_token}',
    }
    response = requests.get(url, headers=headers, params={'api_key': api_key})
    
    if response.status_code == 200:
        return response.json()
    else:
        return None

# List to store movie data
movie_data = []

# Fetch data for each movie ID
for movie_id in movie_ids:
    movie_info = fetch_movie_data(movie_id)
    if movie_info:
        movie_data.append(movie_info)


df = pd.DataFrame(movie_data)

# Display the DataFrame (optional)
print(df.head())



# Cleanse and Preprocess data

### Data Preparation & Cleaning 

In [ ]:
#  Drop irrelevant columns
df_cleaned = df.drop(columns=['adult', 'imdb_id', 'original_title', 'video', 'homepage'])

# Evaluate and clean JSON-like columns
# Helper function to extract and clean data from a JSON-like column
def extract_values(json_column):
  # Handle list of dictionaries
    if isinstance(json_column, list):
        return "|".join([item.get('name', '') for item in json_column if isinstance(item, dict) and 'name' in item])
    # Handle single dictionary (e.g., belongs_to_collection)
    elif isinstance(json_column, dict):
        return json_column.get('name')
    # Everything else → None
    return None

# Extract values from the columns that contain JSON-like data
df_cleaned['collection_name'] = df_cleaned['belongs_to_collection'].apply(lambda x: extract_values(x))
df_cleaned['genres'] = df_cleaned['genres'].apply(lambda x: extract_values(x))
df_cleaned['spoken_languages'] = df_cleaned['spoken_languages'].apply(lambda x: extract_values(x))
df_cleaned['production_countries'] = df_cleaned['production_countries'].apply(lambda x: extract_values(x))
df_cleaned['production_companies'] = df_cleaned['production_companies'].apply(lambda x: extract_values(x))

df_cleaned
# Step 3: Inspect extracted columns using value_counts()
print("Collection Name Value Counts:")
print(df_cleaned['collection_name'].value_counts())

print("\nGenres Value Counts:")
print(df_cleaned['genres'].value_counts())

print("\nSpoken Languages Value Counts:")
print(df_cleaned['spoken_languages'].value_counts())

print("\nProduction Countries Value Counts:")
print(df_cleaned['production_countries'].value_counts())

print("\nProduction Companies Value Counts:")
print(df_cleaned['production_companies'].value_counts())

### Handling Missing And Incorrect Data 

In [ ]:
print("\nStep 5: Converting column datatypes...")
num_cols = ['budget', 'id', 'popularity', 'revenue', 'vote_count', 'vote_average', 'runtime']
for col in num_cols:
    if col in df_cleaned.columns:
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Convert release_date to datetime
if 'release_date' in df_cleaned.columns:
    df_cleaned['release_date'] = pd.to_datetime(df_cleaned['release_date'], errors='coerce')

print("Data types after conversion:")
print(df_cleaned[num_cols + ['release_date']].dtypes)

# Step 6: Replace unrealistic values
print("\nStep 6: Replacing unrealistic values...")

# Replace 0 with NaN for budget, revenue, runtime
for col in ['budget', 'revenue', 'runtime']:
    if col in df_cleaned.columns:
        zero_count = (df_cleaned[col] == 0).sum()
        df_cleaned.loc[df_cleaned[col] == 0, col] = pd.NA
        print(f"  - Replaced {zero_count} zero values in '{col}' with NaN")

# Convert budget and revenue to millions USD
if 'budget' in df_cleaned.columns:
    df_cleaned['budget'] = df_cleaned['budget'] / 1_000_000
    print(f"  - Converted budget to millions USD")

if 'revenue' in df_cleaned.columns:
    df_cleaned['revenue'] = df_cleaned['revenue'] / 1_000_000
    print(f"  - Converted revenue to millions USD")

# Handle movies with vote_count = 0
if 'vote_count' in df_cleaned.columns and 'vote_average' in df_cleaned.columns:
    zero_votes = df_cleaned['vote_count'] == 0
    zero_vote_count = zero_votes.sum()
    
    if zero_vote_count > 0:
        # Set vote_average to NaN for movies with 0 votes
        df_cleaned.loc[zero_votes, 'vote_average'] = pd.NA
        print(f"  - Set vote_average to NaN for {zero_vote_count} movies with vote_count=0")

# Replace placeholders in text columns
text_columns = ['overview', 'tagline']
placeholders = ['', 'No Data', 'N/A', 'NA', 'Unknown', 'no data', 'n/a']

for col in text_columns:
    if col in df_cleaned.columns:
        # Replace placeholders with NaN
        df_cleaned[col] = df_cleaned[col].replace(placeholders, pd.NA)
        # Also replace whitespace-only strings
        df_cleaned[col] = df_cleaned[col].replace(r'^\s*$', pd.NA, regex=True)
        nan_count = df_cleaned[col].isna().sum()
        print(f"  - Replaced placeholders in '{col}': {nan_count} NaN values")

# Step 7: Remove duplicates and drop rows with unknown id/title
print("\nStep 7: Removing duplicates and unknown id/title...")
initial_rows = len(df_cleaned)

# Check for duplicates
duplicate_count = df_cleaned.duplicated().sum()
df_cleaned = df_cleaned.drop_duplicates()
print(f"  - Removed {duplicate_count} duplicate rows")

# Drop rows with missing id or title
if 'id' in df_cleaned.columns and 'title' in df_cleaned.columns:
    df_cleaned = df_cleaned.dropna(subset=['id', 'title'])
    rows_removed = initial_rows - len(df_cleaned)
    print(f"  - Removed {rows_removed} rows with missing id or title")

# Step 8: Keep rows with at least 10 non-NaN values
print("\nStep 8: Filtering rows with sufficient data...")
non_nan_counts = df_cleaned.notna().sum(axis=1)
initial_rows = len(df_cleaned)

df_cleaned = df_cleaned[non_nan_counts >= 10]
rows_removed = initial_rows - len(df_cleaned)
print(f"  - Removed {rows_removed} rows with fewer than 10 non-NaN columns")

# Step 9: Filter to Released movies and drop status column
if 'status' in df_cleaned.columns:
    print(f"  - Status value counts:")
    print(df_cleaned['status'].value_counts())
    
    initial_rows = len(df_cleaned)
    df_cleaned = df_cleaned[df_cleaned['status'] == 'Released']
    rows_removed = initial_rows - len(df_cleaned)
    
    df_cleaned = df_cleaned.drop(columns=['status'])
    print(f"  - Removed {rows_removed} non-Released movies")
    print(f"  - Dropped 'status' column")

print("\n" + "="*80)
print("CLEANING SUMMARY")
print("="*80)
print(f"Final dataframe shape: {df_cleaned.shape}")
print(f"\nMissing values per column:")
print(df_cleaned.isnull().sum().sort_values(ascending=False))

print(f"\nData types:")
print(df_cleaned.dtypes)

print(f"\nNumeric column statistics:")
numeric_columns = df_cleaned.select_dtypes(include=[np.number]).columns
print(df_cleaned[numeric_columns].describe())

### Reorder & Finalize DataFrame 

In [ ]:
# %% [markdown]
# ### Step 10: Reorder Columns

# %%
print("Step 10: Reordering columns...")

# Define desired column order
desired_columns = [
    'id', 'title', 'tagline', 'release_date', 'genres', 'collection_name',
    'original_language', 'budget', 'revenue', 'production_companies',
    'production_countries', 'vote_count', 'vote_average', 'popularity',
    'runtime', 'overview', 'spoken_languages', 'poster_path'
]

# Get columns that exist in the dataframe
existing_columns = [col for col in desired_columns if col in df_cleaned.columns]

# Get any additional columns not in desired order (to append at the end)
additional_columns = [col for col in df_cleaned.columns if col not in desired_columns]

# Combine: desired columns (that exist) + any additional columns
final_column_order = existing_columns + additional_columns

# Reorder the dataframe
df_cleaned = df_cleaned[final_column_order]

print(f"  - Reordered columns")
print(f"  - Column order: {df_cleaned.columns.tolist()}")

# %% [markdown]
# ### Step 11: Reset Index

# %%
print("\nStep 11: Resetting index...")
df_cleaned = df_cleaned.reset_index(drop=True)
print(f"  - Index reset successfully")
print(f"  - Final shape: {df_cleaned.shape}")

# %%
# Display final dataframe
print("\nFinal DataFrame Preview:")
df_cleaned.head(10)

# %% [markdown]
# ## Save Cleaned Data

# %%
# Save cleaned data to CSV
df_cleaned.to_csv('tmdb_movies_cleaned.csv', index=False)
print("✅ Cleaned data saved to 'tmdb_movies_cleaned.csv'")
print(f"   Shape: {df_cleaned.shape}")
print(f"   Columns: {len(df_cleaned.columns)}")
print(f"   Rows: {len(df_cleaned)}")

#  KPI Implementation & Analysis

In [ ]:
# %% [markdown]
# ## Identify Best/Worst Performing Movies

# %% [markdown]
# ### Create Derived Columns (Profit & ROI)

# %%
print("Creating derived columns for analysis...")

# Calculate Profit (Revenue - Budget) in millions USD
df_cleaned['profit'] = df_cleaned['revenue'] - df_cleaned['budget']

# Calculate ROI (Return on Investment) = Revenue / Budget
# Only calculate for movies with budget > 0 to avoid division by zero
df_cleaned['roi'] = df_cleaned.apply(
    lambda row: row['revenue'] / row['budget'] if pd.notna(row['budget']) and row['budget'] > 0 else np.nan,
    axis=1
)

print(f"✅ Added 'profit' column (Revenue - Budget)")
print(f"✅ Added 'roi' column (Revenue / Budget)")
print(f"\nProfit range: ${df_cleaned['profit'].min():.2f}M to ${df_cleaned['profit'].max():.2f}M")
print(f"ROI range: {df_cleaned['roi'].min():.2f}x to {df_cleaned['roi'].max():.2f}x")

# %% [markdown]
# ### Define User-Defined Function (UDF) for Ranking

# %%
def rank_movies(df, column, top_n=10, ascending=False, filter_condition=None, 
                display_columns=None, rank_name=None):
    """
    User-Defined Function to rank and display top/bottom movies.
    
    Parameters:
    -----------
    df : DataFrame
        The movies dataframe
    column : str
        Column name to rank by
    top_n : int, default=10
        Number of top movies to return
    ascending : bool, default=False
        If True, rank from lowest to highest (worst performers)
        If False, rank from highest to lowest (best performers)
    filter_condition : pandas Series or None
        Boolean mask to filter dataframe before ranking
    display_columns : list or None
        Columns to display in output. If None, uses default set
    rank_name : str or None
        Custom name for the ranking (for display purposes)
    
    Returns:
    --------
    DataFrame : Top N ranked movies
    """
    # Apply filter if provided
    if filter_condition is not None:
        df_filtered = df[filter_condition].copy()
    else:
        df_filtered = df.copy()
    
    # Remove rows where the ranking column is NaN
    df_filtered = df_filtered.dropna(subset=[column])
    
    # Sort by the specified column
    df_sorted = df_filtered.sort_values(by=column, ascending=ascending)
    
    # Get top N
    top_movies = df_sorted.head(top_n)
    
    # Define default display columns if not provided
    if display_columns is None:
        display_columns = ['title', 'release_date', column, 'budget', 'revenue']
    
    # Ensure all display columns exist
    display_columns = [col for col in display_columns if col in top_movies.columns]
    
    # Create rank name for display
    if rank_name is None:
        direction = "Lowest" if ascending else "Highest"
        rank_name = f"{direction} {column.replace('_', ' ').title()}"
    
    # Display results
    print("\n" + "="*80)
    print(f"{rank_name} (Top {len(top_movies)})")
    print("="*80)
    
    return top_movies[display_columns]

print("✅ User-Defined Function 'rank_movies()' defined successfully!")

# %% [markdown]
# ### 1. Highest Revenue Movies

# %%
highest_revenue = rank_movies(
    df_cleaned,
    column='revenue',
    top_n=10,
    ascending=False,
    display_columns=['title', 'release_date', 'revenue', 'budget', 'profit'],
    rank_name="🏆 Highest Revenue Movies"
)
highest_revenue

# %% [markdown]
# ### 2. Highest Budget Movies

# %%
highest_budget = rank_movies(
    df_cleaned,
    column='budget',
    top_n=10,
    ascending=False,
    display_columns=['title', 'release_date', 'budget', 'revenue', 'profit'],
    rank_name="💰 Highest Budget Movies"
)
highest_budget

# %% [markdown]
# ### 3. Highest Profit Movies

# %%
highest_profit = rank_movies(
    df_cleaned,
    column='profit',
    top_n=10,
    ascending=False,
    display_columns=['title', 'release_date', 'profit', 'revenue', 'budget', 'roi'],
    rank_name="📈 Highest Profit Movies (Revenue - Budget)"
)
highest_profit

# %% [markdown]
# ### 4. Lowest Profit Movies (Biggest Losses)

# %%
lowest_profit = rank_movies(
    df_cleaned,
    column='profit',
    top_n=10,
    ascending=True,
    display_columns=['title', 'release_date', 'profit', 'revenue', 'budget', 'roi'],
    rank_name="📉 Lowest Profit Movies (Biggest Losses)"
)
lowest_profit

# %% [markdown]
# ### 5. Highest ROI Movies (Budget ≥ $10M)

# %%
# Filter: Budget >= 10 million USD
budget_threshold = 10

highest_roi = rank_movies(
    df_cleaned,
    column='roi',
    top_n=10,
    ascending=False,
    filter_condition=(df_cleaned['budget'] >= budget_threshold),
    display_columns=['title', 'release_date', 'roi', 'budget', 'revenue', 'profit'],
    rank_name=f"🚀 Highest ROI Movies (Budget ≥ ${budget_threshold}M)"
)
highest_roi

# %% [markdown]
# ### 6. Lowest ROI Movies (Budget ≥ $10M)

# %%
lowest_roi = rank_movies(
    df_cleaned,
    column='roi',
    top_n=10,
    ascending=True,
    filter_condition=(df_cleaned['budget'] >= budget_threshold),
    display_columns=['title', 'release_date', 'roi', 'budget', 'revenue', 'profit'],
    rank_name=f"💸 Lowest ROI Movies (Budget ≥ ${budget_threshold}M)"
)
lowest_roi

# %% [markdown]
# ### 7. Most Voted Movies

# %%
most_voted = rank_movies(
    df_cleaned,
    column='vote_count',
    top_n=10,
    ascending=False,
    display_columns=['title', 'release_date', 'vote_count', 'vote_average', 'popularity'],
    rank_name="👥 Most Voted Movies"
)
most_voted

# %% [markdown]
# ### 8. Highest Rated Movies (≥10 votes)

# %%
# Filter: vote_count >= 10
vote_threshold = 10

highest_rated = rank_movies(
    df_cleaned,
    column='vote_average',
    top_n=10,
    ascending=False,
    filter_condition=(df_cleaned['vote_count'] >= vote_threshold),
    display_columns=['title', 'release_date', 'vote_average', 'vote_count', 'popularity'],
    rank_name=f"⭐ Highest Rated Movies (≥{vote_threshold} votes)"
)
highest_rated

# %% [markdown]
# ### 9. Lowest Rated Movies (≥10 votes)

# %%
lowest_rated = rank_movies(
    df_cleaned,
    column='vote_average',
    top_n=10,
    ascending=True,
    filter_condition=(df_cleaned['vote_count'] >= vote_threshold),
    display_columns=['title', 'release_date', 'vote_average', 'vote_count', 'popularity'],
    rank_name=f"⭐ Lowest Rated Movies (≥{vote_threshold} votes)"
)
lowest_rated

# %% [markdown]
# ### 10. Most Popular Movies

# %%
most_popular = rank_movies(
    df_cleaned,
    column='popularity',
    top_n=10,
    ascending=False,
    display_columns=['title', 'release_date', 'popularity', 'vote_count', 'vote_average'],
    rank_name="🔥 Most Popular Movies"
)
most_popular

# %% [markdown]
# ### Summary of All Rankings

# %%
print("\n" + "="*80)
print("📊 RANKING SUMMARY")
print("="*80)

rankings = {
    "Highest Revenue": highest_revenue['title'].iloc[0] if len(highest_revenue) > 0 else "N/A",
    "Highest Budget": highest_budget['title'].iloc[0] if len(highest_budget) > 0 else "N/A",
    "Highest Profit": highest_profit['title'].iloc[0] if len(highest_profit) > 0 else "N/A",
    "Biggest Loss": lowest_profit['title'].iloc[0] if len(lowest_profit) > 0 else "N/A",
    "Highest ROI": highest_roi['title'].iloc[0] if len(highest_roi) > 0 else "N/A",
    "Lowest ROI": lowest_roi['title'].iloc[0] if len(lowest_roi) > 0 else "N/A",
    "Most Voted": most_voted['title'].iloc[0] if len(most_voted) > 0 else "N/A",
    "Highest Rated": highest_rated['title'].iloc[0] if len(highest_rated) > 0 else "N/A",
    "Lowest Rated": lowest_rated['title'].iloc[0] if len(lowest_rated) > 0 else "N/A",
    "Most Popular": most_popular['title'].iloc[0] if len(most_popular) > 0 else "N/A"
}

for category, movie in rankings.items():
    print(f"{category:20s}: {movie}")

print("="*80)

# %% [markdown]
# ### Export Rankings to Excel (Optional)

# %%
# Uncomment to export all rankings to Excel with separate sheets
"""
with pd.ExcelWriter('movie_rankings.xlsx', engine='openpyxl') as writer:
    highest_revenue.to_excel(writer, sheet_name='Highest Revenue', index=False)
    highest_budget.to_excel(writer, sheet_name='Highest Budget', index=False)
    highest_profit.to_excel(writer, sheet_name='Highest Profit', index=False)
    lowest_profit.to_excel(writer, sheet_name='Lowest Profit', index=False)
    highest_roi.to_excel(writer, sheet_name='Highest ROI', index=False)
    lowest_roi.to_excel(writer, sheet_name='Lowest ROI', index=False)
    most_voted.to_excel(writer, sheet_name='Most Voted', index=False)
    highest_rated.to_excel(writer, sheet_name='Highest Rated', index=False)
    lowest_rated.to_excel(writer, sheet_name='Lowest Rated', index=False)
    most_popular.to_excel(writer, sheet_name='Most Popular', index=False)

print("✅ Rankings exported to 'movie_rankings.xlsx'")
"""

# Advanced Movie Filtering & Search Queries 

### Filter the dataset for specific queries: 

In [ ]:
# %% [markdown]
# ## Advanced Movie Filtering & Search Queries

# %% [markdown]
# ### Fetch Credits Data from TMDB API

# %%
def fetch_movie_credits(movie_id, access_token):
    """Fetch cast and director from TMDB credits endpoint."""
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits"
    headers = {'Authorization': f'Bearer {access_token}'}
    
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        
        # Extract top 5 cast members
        cast_list = data.get('cast', [])
        cast_names = "|".join([actor['name'] for actor in cast_list[:5]])
        
        # Extract director
        crew_list = data.get('crew', [])
        director = None
        for person in crew_list:
            if person.get('job') == 'Director':
                director = person.get('name')
                break
        
        return {
            'cast': cast_names if cast_names else None,
            'director': director,
            'cast_size': len(cast_list)
        }
    else:
        return {'cast': None, 'director': None, 'cast_size': 0}

# Fetch credits for all movies
print("Fetching credits data...")
credits_data = []
for movie_id in df_cleaned['id']:
    credits = fetch_movie_credits(movie_id, access_token)
    credits['id'] = movie_id
    credits_data.append(credits)

# Merge credits into main dataframe
df_credits = pd.DataFrame(credits_data)
print("Credits DataFrame:")
print(df_credits)
df_cleaned = df_cleaned.merge(df_credits, on='id', how='left')

print(f"✅ Credits data added: {len(df_cleaned)} movies")

# %% [markdown]
# ### Define Search Function

# %%
def search_movies(df, genres=None, cast_member=None, director=None, 
                  sort_by='vote_average', ascending=False):
    """
    Filter movies by genres, cast, and director.
    
    Parameters:
    - genres: list of genre names
    - cast_member: actor/actress name
    - director: director name
    - sort_by: column to sort by
    - ascending: sort order
    """
    result = df.copy()
    
    # Filter by genres
    if genres:
        if isinstance(genres, str):
            genres = [genres]
        genre_mask = result['genres'].apply(
            lambda x: any(g in str(x) for g in genres) if pd.notna(x) else False
        )
        result = result[genre_mask]
    
    # Filter by cast
    if cast_member:
        result = result[result['cast'].str.contains(cast_member, case=False, na=False)]
    
    # Filter by director
    if director:
        result = result[result['director'].str.contains(director, case=False, na=False)]
    
    # Sort results
    result = result.dropna(subset=[sort_by])
    result = result.sort_values(by=sort_by, ascending=ascending)
    
    return result

# %% [markdown]
# ### Search 1: Best-Rated Sci-Fi Action Movies with Bruce Willis

# %%
print("="*80)
print("SEARCH 1: Science Fiction + Action Movies Starring Bruce Willis")
print("="*80)

search1 = search_movies(
    df_cleaned,
    genres=['Science Fiction', 'Action'],
    cast_member='Bruce Willis',
    sort_by='vote_average',
    ascending=False
)

display_cols = ['title', 'release_date', 'genres', 'vote_average', 'cast', 'director']
print(f"\nFound {len(search1)} movies\n")
search1[display_cols]

# %% [markdown]
# ### Search 2: Uma Thurman + Quentin Tarantino (by Runtime)

# %%
print("="*80)
print("SEARCH 2: Uma Thurman Movies Directed by Quentin Tarantino")
print("="*80)

search2 = search_movies(
    df_cleaned,
    cast_member='Uma Thurman',
    director='Quentin Tarantino',
    sort_by='runtime',
    ascending=True
)

display_cols = ['title', 'release_date', 'runtime', 'vote_average', 'cast', 'director']
print(f"\nFound {len(search2)} movies (sorted by runtime - shortest to longest)\n")
search2[display_cols]

# Franchise vs. Standalone Movie Performance 

### Compare movie franchises (collection_name) vs. standalone movies

In [ ]:
# %% [markdown]
# ## Compare Franchise vs Standalone Movies

# %% [markdown]
# ### Create Franchise Classification

# %%
# Create a binary column to identify franchise vs standalone movies
df_cleaned['is_franchise'] = df_cleaned['collection_name'].notna()

# Count movies in each category
franchise_count = df_cleaned['is_franchise'].sum()
standalone_count = (~df_cleaned['is_franchise']).sum()

print("Movie Classification:")
print(f"  Franchise Movies: {franchise_count}")
print(f"  Standalone Movies: {standalone_count}")
print(f"  Total: {len(df_cleaned)}")

# %% [markdown]
# ### Calculate Comparison Metrics

# %%
# Group by franchise status and calculate metrics
comparison = df_cleaned.groupby('is_franchise').agg({
    'revenue': 'mean',           # Mean Revenue
    'roi': 'median',              # Median ROI
    'budget': 'mean',             # Mean Budget
    'popularity': 'mean',         # Mean Popularity
    'vote_average': 'mean'        # Mean Rating
}).round(2)

# Rename index for clarity
comparison.index = ['Standalone', 'Franchise']

# Rename columns for better display
comparison.columns = [
    'Mean Revenue ($M)',
    'Median ROI',
    'Mean Budget ($M)',
    'Mean Popularity',
    'Mean Rating'
]

print("\n" + "="*80)
print("FRANCHISE vs STANDALONE MOVIES COMPARISON")
print("="*80)
print(comparison)

# %% [markdown]
# ### Visualize the Comparison

# %%
# Calculate percentage differences
standalone_vals = comparison.loc['Standalone']
franchise_vals = comparison.loc['Franchise']

print("\n" + "="*80)
print("PERCENTAGE DIFFERENCES (Franchise vs Standalone)")
print("="*80)

for col in comparison.columns:
    standalone = standalone_vals[col]
    franchise = franchise_vals[col]
    
    if pd.notna(standalone) and pd.notna(franchise) and standalone != 0:
        pct_diff = ((franchise - standalone) / standalone) * 100
        direction = "higher" if pct_diff > 0 else "lower"
        print(f"{col:25s}: {abs(pct_diff):6.1f}% {direction}")
    else:
        print(f"{col:25s}: N/A")

# %% [markdown]
# ### Detailed Statistics

# %%
print("\n" + "="*80)
print("DETAILED STATISTICS")
print("="*80)

# Create more detailed comparison
detailed_stats = df_cleaned.groupby('is_franchise').agg({
    'revenue': ['mean', 'median', 'std', 'min', 'max'],
    'roi': ['mean', 'median', 'std', 'min', 'max'],
    'budget': ['mean', 'median', 'std', 'min', 'max'],
    'popularity': ['mean', 'median', 'std', 'min', 'max'],
    'vote_average': ['mean', 'median', 'std', 'min', 'max']
}).round(2)

detailed_stats.index = ['Standalone', 'Franchise']

print("\nRevenue Statistics ($M):")
print(detailed_stats['revenue'])

print("\nROI Statistics:")
print(detailed_stats['roi'])

print("\nBudget Statistics ($M):")
print(detailed_stats['budget'])

print("\nPopularity Statistics:")
print(detailed_stats['popularity'])

print("\nRating Statistics:")
print(detailed_stats['vote_average'])

# %% [markdown]
# ### Summary Insights

# %%
print("\n" + "="*80)
print("KEY INSIGHTS")
print("="*80)

# Determine which performs better in each category
insights = []

for metric in ['Mean Revenue ($M)', 'Median ROI', 'Mean Budget ($M)', 
               'Mean Popularity', 'Mean Rating']:
    standalone_val = comparison.loc['Standalone', metric]
    franchise_val = comparison.loc['Franchise', metric]
    
    if pd.notna(standalone_val) and pd.notna(franchise_val):
        winner = 'Franchise' if franchise_val > standalone_val else 'Standalone'
        diff = abs(franchise_val - standalone_val)
        insights.append(f"✓ {metric}: {winner} wins by {diff:.2f}")

for insight in insights:
    print(insight)

# %% [markdown]
# ### Export Comparison Results

# %%
# Create summary dataframe for export
summary_df = comparison.copy()
summary_df['Movie Count'] = [standalone_count, franchise_count]

# Reorder columns to show count first
cols = ['Movie Count'] + [col for col in summary_df.columns if col != 'Movie Count']
summary_df = summary_df[cols]



print("\n" + "="*80)
print("FINAL SUMMARY TABLE")
print("="*80)
print(summary_df)

# Uncomment to save to CSV
summary_df.to_csv('franchise_vs_standalone_comparison.csv')
print("\n✅ Comparison exported to 'franchise_vs_standalone_comparison.csv'")

### Most Successful Franchises & Directors

In [ ]:
# %% [markdown]
# ## Most Successful Franchises & Directors

# %% [markdown]
# ### 4. Most Successful Movie Franchises

# %%
franchise_stats = df_cleaned[df_cleaned['collection_name'].notna()].groupby('collection_name').agg({
    'id': 'count',
    'budget': ['sum', 'mean'],
    'revenue': ['sum', 'mean'],
    'vote_average': 'mean'
}).round(2)

franchise_stats.columns = ['Movie Count', 'Total Budget ($M)', 'Mean Budget ($M)', 
                           'Total Revenue ($M)', 'Mean Revenue ($M)', 'Mean Rating']
franchise_stats = franchise_stats.reset_index().sort_values('Total Revenue ($M)', ascending=False)

print("MOST SUCCESSFUL FRANCHISES")
print("="*80)
print(franchise_stats)  

# %% [markdown]
# ### 5. Most Successful Directors

# %%
director_stats = df_cleaned[df_cleaned['director'].notna()].groupby('director').agg({
    'id': 'count',
    'revenue': 'sum',
    'vote_average': 'mean'
}).round(2)

director_stats.columns = ['Movies Directed', 'Total Revenue ($M)', 'Mean Rating']
director_stats = director_stats.reset_index().sort_values('Total Revenue ($M)', ascending=False)

print("MOST SUCCESSFUL DIRECTORS")
print("="*80)
director_stats

# Data Visualization

In [ ]:
# Data Visualization

In [ ]:
# %% [markdown]
# ## Data Visualization

# %%

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# %% [markdown]
# ### 1. Revenue vs. Budget Trends

# %%
plt.figure(figsize=(12, 6))
plt.scatter(df_cleaned['budget'], df_cleaned['revenue'], alpha=0.6, s=50)
plt.xlabel('Budget ($M)', fontsize=12)
plt.ylabel('Revenue ($M)', fontsize=12)
plt.title('Revenue vs Budget', fontsize=14, fontweight='bold')

# Add diagonal line (break-even line)
max_val = max(df_cleaned['budget'].max(), df_cleaned['revenue'].max())
plt.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Break-even line')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# %% [markdown]
# ### 2. ROI Distribution by Genre

# %%
# Extract primary genre (first genre in the list)
df_cleaned['primary_genre'] = df_cleaned['genres'].str.split('|').str[0]

# Filter valid ROI values
df_cleaned.info()
roi_by_genre = df_cleaned[df_cleaned['roi'].notna()].groupby('primary_genre')['roi'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
roi_by_genre.plot(kind='bar', color='steelblue')
plt.xlabel('Genre', fontsize=12)
plt.ylabel('Average ROI', fontsize=12)
plt.title('ROI Distribution by Genre', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# %% [markdown]
# ### 3. Popularity vs. Rating

# %%
plt.figure(figsize=(12, 6))
plt.scatter(df_cleaned['vote_average'], df_cleaned['popularity'], alpha=0.6, s=50, c=df_cleaned['revenue'], cmap='viridis')
plt.xlabel('Rating (Vote Average)', fontsize=12)
plt.ylabel('Popularity', fontsize=12)
plt.title('Popularity vs Rating (colored by Revenue)', fontsize=14, fontweight='bold')
plt.colorbar(label='Revenue ($M)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# %% [markdown]
# ### 4. Yearly Trends in Box Office Performance

# %%
# Extract year from release_date
df_cleaned['year'] = pd.to_datetime(df_cleaned['release_date']).dt.year

# Group by year
yearly_stats = df_cleaned.groupby('year').agg({
    'revenue': 'mean',
    'budget': 'mean',
    'id': 'count'
}).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 6))

# Revenue and Budget
ax1.plot(yearly_stats['year'], yearly_stats['revenue'], marker='o', linewidth=2, label='Avg Revenue', color='green')
ax1.plot(yearly_stats['year'], yearly_stats['budget'], marker='s', linewidth=2, label='Avg Budget', color='blue')
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Amount ($M)', fontsize=12)
ax1.tick_params(axis='y')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Number of movies (secondary axis)
ax2 = ax1.twinx()
ax2.bar(yearly_stats['year'], yearly_stats['id'], alpha=0.3, color='orange', label='Movie Count')
ax2.set_ylabel('Number of Movies', fontsize=12)
ax2.legend(loc='upper right')

plt.title('Yearly Trends in Box Office Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# %% [markdown]
# ### 5. Franchise vs. Standalone Success Comparison

# %%
comparison_data = df_cleaned.groupby('is_franchise').agg({
    'revenue': 'mean',
    'budget': 'mean',
    'roi': 'median',
    'vote_average': 'mean',
    'popularity': 'mean'
}).reset_index()

comparison_data['is_franchise'] = comparison_data['is_franchise'].map({False: 'Standalone', True: 'Franchise'})

# Create subplots
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Franchise vs Standalone Movies Comparison', fontsize=16, fontweight='bold')

metrics = [
    ('revenue', 'Mean Revenue ($M)', axes[0, 0]),
    ('budget', 'Mean Budget ($M)', axes[0, 1]),
    ('roi', 'Median ROI', axes[0, 2]),
    ('vote_average', 'Mean Rating', axes[1, 0]),
    ('popularity', 'Mean Popularity', axes[1, 1])
]

for col, title, ax in metrics:
    comparison_data.plot(x='is_franchise', y=col, kind='bar', ax=ax, legend=False, color=['steelblue', 'coral'])
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(title)
    ax.tick_params(axis='x', rotation=0)
    ax.grid(axis='y', alpha=0.3)

# Remove extra subplot
fig.delaxes(axes[1, 2])

plt.tight_layout()
plt.show()

# %% [markdown]
# ### Summary Statistics Table

# %%
print("\n" + "="*80)
print("VISUALIZATION SUMMARY STATISTICS")
print("="*80)

print("\nRevenue vs Budget Correlation:")
correlation = df_cleaned[['budget', 'revenue']].corr().iloc[0, 1]
print(f"  Correlation coefficient: {correlation:.3f}")

print("\nTop 3 Genres by ROI:")
print(roi_by_genre.head(3))

print("\nYearly Performance Range:")
print(f"  Years covered: {yearly_stats['year'].min():.0f} - {yearly_stats['year'].max():.0f}")
print(f"  Highest revenue year: {yearly_stats.loc[yearly_stats['revenue'].idxmax(), 'year']:.0f}")
print(f"  Most productive year: {yearly_stats.loc[yearly_stats['id'].idxmax(), 'year']:.0f}")